In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from openf1_client import get


data = pd.read_csv("../data/processed/dataset_2026.csv")
data["quali_pos"] = data["quali_pos"].fillna(20)    #no quali_time = back of the grid

race_dates = get("sessions", year = 2026, session_name = "Race")[["location", "date_start"]]
data = data.merge(race_dates, on = "location")
order = data.sort_values("date_start")["location"].unique()

print(order)

<StringArray>
[   'Melbourne',       'Suzuka',  'Monte Carlo',    'Barcelona',
    'Spielberg',     'Budapest',        'Monza',       'Madrid',
 'Kuala Lumpur',       'Austin',  'Mexico City',    'São Paulo']
Length: 12, dtype: str


In [6]:
def walk_forward(data, features, alpha = 1.0, min_train = 3):   #the model has min 3 races for learning
    results = []
    for i in range(min_train, len(order)):
        train = data[data["location"].isin(order[:i])]  #all races before
        test = data[data["location"] == order[i]]   #the race which the model predicts

        model = Ridge(alpha = alpha).fit(train[features], train["race_gap_pct"])
        pred = model.predict(test[features])

        results.append({
            "location": order[i],
            "rho": pd.Series(pred, index = test.index).corr(test["race_gap_pct"], method = "spearman"),
            "mae": np.mean(np.abs(pred - test["race_gap_pct"])) #mean absolute error
        })

    return pd.DataFrame(results)


In [7]:
feature_sets = {
    "fp2_only": ["fp2_gap_pct"],                #Friday prediction
    "quali_only": ["quali_pos"],                #baseline
    "fp2_plus_quali": ["fp2_gap_pct", "quali_pos"]  #Saturday prediction
}

summary = {}

for name, feats in feature_sets.items():
    res = walk_forward(data, feats)
    summary[name] = res[["rho", "mae"]].mean()

pd.DataFrame(summary).T

,rho,mae
fp2_only,0.817194,0.564332
quali_only,0.861369,0.598869
fp2_plus_quali,0.886436,0.453079


## Model: Ridge regression with walk-forward validation

**Setup.** Target: race_gap_pct (corrected race pace, % gap to the fastest driver).
Three Ridge models (alpha = 1.0), each with a different set of inputs:
- fp2_only: fp2_gap_pct -> the prediction available on Friday, after FP2
- quali_only: quali_pos -> baseline, what everybody knows after qualifying
- fp2_plus_quali: both -> the prediction available on Saturday

Drivers without a qualifying time get quali_pos = 20 (back of the grid).

**Validation.** Walk-forward over the 12 usable 2026 races: for each race, the model
is trained only on the races before it and tested on that race (first 3 races used
only for training, 9 test races), avoiding data leakage.

**Conclusion.**
- Combining FP2 and qualifying gives the best order and reduces the error by about 24%
  compared to qualifying alone. So FP2 long runs contain information that contributes to improving the model's prediction.